# Module 7: Causal Inference for Child Welfare Policy
**ACS Predictive Analytics Curriculum**

This is the most sophisticated module. It answers:
*Does investigative consultation actually help children?*

Concepts covered:
- Why correlation ≠ causation in ACS data
- Regression Discontinuity Design (RDD) at risk threshold
- Propensity Score Matching (PSM)
- Difference-in-Differences (DiD) for policy changes
- Instrumental Variables (IV)


In [ ]:
install.packages(c('rdrobust','MatchIt','did','AER','tidyverse'),
                 repos='https://cran.rstudio.com/', quiet=TRUE)
library(tidyverse)
library(rdrobust)
library(MatchIt)
library(did)
library(AER)

features <- read_csv('data/acs_features.csv', show_col_types=FALSE)
scr      <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE) %>%
  mutate(report_date=as.Date(report_date))
cat('Packages and data loaded\n')

## SECTION 7.1: The Fundamental Problem — Why OLS Fails Here

**The naive question:** Do cases that receive investigative consultation have better outcomes?

**The problem:** Consultation is assigned to HARDER cases.
So worse cases get consultation → if we compare outcomes naively,
consultation looks harmful. This is confounding.

```
True causal path:
High risk case → gets consultation → better outcome

Naive comparison sees:
Consultation cases: worse outcomes (because they WERE worse cases)
No consultation:    better outcomes (because they WERE easier cases)
Conclusion: consultation is harmful ← WRONG
```

We need causal inference to untangle this.

In [ ]:
# Simulate the confounding problem
set.seed(42)

# Create simulated outcome data
causal_data <- features %>%
  mutate(
    # Simplified risk score
    risk_score = scale(prior_reports_12mo * 0.4 +
                       prior_substantiated_flag * 0.8 +
                       dv_history_flag * 0.5 +
                       shelter_involvement_flag * 0.3)[,1],

    # Threshold: cases above 0 get consultation (simplified)
    got_consultation = needs_investigative_consultation == 1,

    # Simulate 90-day outcome: new_report (1=bad, 0=good)
    # True effect: consultation reduces new reports by 15%
    baseline_prob = plogis(risk_score * 0.5 + 0.2),
    new_report_90d = rbinom(n(), 1,
      baseline_prob - 0.15 * got_consultation + rnorm(n(), 0, 0.05))
  )

# Naive comparison - WRONG
cat('NAIVE (CONFOUNDED) COMPARISON:\n')
causal_data %>%
  group_by(got_consultation) %>%
  summarise(
    n=n(),
    new_report_rate=round(mean(new_report_90d)*100, 1)
  )

# Naive says consultation is associated with MORE new reports
# But that's because harder cases got consultation
cat('\nNote: Consultation cases have MORE new reports naively')
cat('\nBut this is confounding - harder cases got consultation')
cat('\nTrue effect is -15% which we cannot see without causal methods\n')

## SECTION 7.2: Regression Discontinuity Design (RDD)

In [ ]:
# RDD exploits a THRESHOLD in assignment
# ACS assigns consultation based on risk score
# Cases just above threshold ≈ cases just below (similar risk)
# Difference in outcomes = causal effect of consultation

# Simulate continuous risk score with threshold at 0.5
rdd_data <- causal_data %>%
  mutate(
    risk_score_continuous = plogis(risk_score),  # 0-1 scale
    above_threshold       = risk_score_continuous >= 0.5,
    # Sharper treatment assignment at threshold
    got_consultation_rdd  = risk_score_continuous >= 0.5
  )

# Visualize the discontinuity
rdd_data %>%
  mutate(bin = cut(risk_score_continuous, breaks=20)) %>%
  group_by(bin, got_consultation_rdd) %>%
  summarise(
    mid    = mean(risk_score_continuous),
    outcome= mean(new_report_90d),
    .groups='drop'
  ) %>%
  ggplot(aes(x=mid, y=outcome, color=got_consultation_rdd)) +
  geom_point(size=3) +
  geom_smooth(method='lm', se=TRUE) +
  geom_vline(xintercept=0.5, linetype='dashed', linewidth=1.2, color='black') +
  annotate('text', x=0.52, y=0.7, label='Threshold\n(consultation assigned)',
           hjust=0, size=4) +
  scale_color_manual(values=c('FALSE'='#2171B5','TRUE'='#CB181D'),
                     labels=c('No Consultation','Got Consultation')) +
  labs(title='Regression Discontinuity: Effect of Consultation on Outcomes',
       subtitle='Jump at threshold = causal effect of consultation',
       x='Risk Score', y='Pr(New Report in 90 Days)', color=NULL) +
  theme_minimal(base_size=12)

In [ ]:
# Formal RDD estimation with rdrobust
rdd_result <- rdrobust(
  y = rdd_data$new_report_90d,
  x = rdd_data$risk_score_continuous,
  c = 0.5  # threshold
)

summary(rdd_result)

cat('\nINTERPRETATION:')
cat('\nThe coefficient is the CAUSAL effect of crossing the threshold')
cat('\n(i.e., receiving consultation) on 90-day new report probability')
cat('\nNegative = consultation reduces new reports (what we hope to see)')
cat('\nThis is valid ONLY near the threshold - not for very high/low cases')

## SECTION 7.3: Propensity Score Matching

In [ ]:
# PSM: Match each treated case to a similar untreated case
# Then compare outcomes within matched pairs
# Removes confounding by severity

psm_data <- causal_data %>%
  select(report_id=report_id, got_consultation,
         new_report_90d, prior_reports_12mo, prior_substantiated_flag,
         dv_history_flag, shelter_involvement_flag, caseworker_caseload,
         n_children_in_household, reporter_accuracy_score) %>%
  drop_na()

# Estimate propensity scores
match_out <- matchit(
  got_consultation ~ prior_reports_12mo + prior_substantiated_flag +
    dv_history_flag + shelter_involvement_flag + n_children_in_household,
  data   = psm_data,
  method = 'nearest',   # nearest neighbor matching
  distance= 'logit',    # propensity score via logistic regression
  ratio  = 1            # 1:1 matching
)

summary(match_out)

In [ ]:
# Extract matched data and compare outcomes
matched_data <- match.data(match_out)

cat('PROPENSITY SCORE MATCHED COMPARISON:\n')
matched_data %>%
  group_by(got_consultation) %>%
  summarise(
    n               = n(),
    new_report_rate = round(mean(new_report_90d)*100, 1),
    avg_prior_reports=round(mean(prior_reports_12mo), 2)
  )

# OLS on matched data
matched_model <- lm(new_report_90d ~ got_consultation +
                     prior_reports_12mo + dv_history_flag,
                   data=matched_data, weights=weights)
summary(matched_model)

cat('\nINTERPRETATION:')
cat('\ngot_consultationTRUE coefficient = causal effect of consultation')
cat('\nMatched cases are similar in severity')
cat('\nSo difference in outcomes is closer to causal')

## SECTION 7.4: Difference in Differences

In [ ]:
# DiD: Compare change over time in treated vs control group
# Useful when ACS rolls out a new policy in some boroughs

# Simulate: ACS implements new reporter training in Bronx (2023)
did_data <- scr %>%
  mutate(
    treated     = borough == 'Bronx',
    post        = year(report_date) >= 2023,
    # Simulate outcome: monthly report volume
    month_year  = floor_date(report_date, 'month'),
    # True effect: training reduces reports by 10%
    report_volume= 1  # each row = one report
  ) %>%
  count(borough, month_year, treated, post, name='n_reports')

# DiD estimate
did_model <- lm(n_reports ~ treated * post, data=did_data)
summary(did_model)

cat('\nINTERPRETATION:')
cat('\ntreatedTRUE:postTRUE = DiD estimate')
cat('\n= (Bronx after - Bronx before) - (other boroughs after - before)')
cat('\nRemoves time trends that affect all boroughs equally')
cat('\nIsolates the effect of the training program')

## SECTION 7.5: Assumptions and Limitations

In [ ]:
cat('CAUSAL INFERENCE ASSUMPTIONS BY METHOD:\n\n')

cat('RDD Assumptions:')
cat('\n  1. No manipulation of running variable at threshold')
cat('\n     (families cannot adjust their risk score to get consultation)')
cat('\n  2. Continuity: potential outcomes continuous at threshold')
cat('\n  3. Only valid near the threshold (external validity limited)')
cat('\n  Test: rddensity() - checks for bunching at threshold\n\n')

cat('PSM Assumptions:')
cat('\n  1. Conditional independence: no unmeasured confounders')
cat('\n     (we measured all variables that affect both treatment and outcome)')
cat('\n  2. Common support: overlap in propensity scores')
cat('\n  3. Sensitive to unmeasured confounders (unlike RDD)\n\n')

cat('DiD Assumptions:')
cat('\n  1. Parallel trends: Bronx and other boroughs would have')
cat('\n     followed same trend without the training program')
cat('\n  2. No spillover: training did not affect non-Bronx reporters')
cat('\n  3. Test parallel trends with pre-treatment data\n\n')

cat('RECOMMENDATION FOR ACS:')
cat('\n  RDD is most credible because threshold is externally determined')
cat('\n  Cases near threshold are as-good-as-randomly assigned')
cat('\n  Start there before PSM or DiD')